In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import mudata
from scipy.sparse import csr_matrix
import warnings
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3
warnings.filterwarnings("ignore")
plt.rcParams['pdf.fonttype']=42

In [ ]:
#resegmented

bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs4_reseg_snm3_2T_L3.h5ad")

bgs4

In [ ]:
bgs4_lge_mge = bgs4

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Use your AnnData object
adata = bgs4  # or bgs4_lge_mge if that's the one you're using

# Get cluster annotation
clusters = adata.obs['adjusted_L3_transferred']
cluster_str = clusters.astype(str)

# Exclude 'uncertain' cells entirely
mask_known = cluster_str != 'uncertain'
cluster_str_filtered = cluster_str[mask_known]

# Get unique known clusters and sort nicely (numbers first, then others)
numeric = pd.to_numeric(clusters[mask_known], errors='coerce')
is_numeric = numeric.notna()
numeric_unique = np.sort(numeric[is_numeric].unique().astype(int))
non_numeric_unique = np.sort(clusters[mask_known & ~is_numeric].unique())

all_unique = [str(c) for c in numeric_unique] + [str(c) for c in non_numeric_unique]

# Use pre-defined colors if available (recommended!)
if ('adjusted_L3_transferred' in adata.obs and
    adata.obs['adjusted_L3_transferred'].dtype.name == 'category' and
    'adjusted_L3_transferred_colors' in adata.uns):

    categories = adata.obs['adjusted_L3_transferred'].cat.categories
    known_categories = [cat for cat in categories if str(cat) != 'uncertain']
    known_categories_str = [str(cat) for cat in known_categories]
    color_list = adata.uns['adjusted_L3_transferred_colors']
    # Match colors to known categories
    cat_to_color = dict(zip([str(cat) for cat in categories], color_list))
    cluster_colors = {clust: cat_to_color.get(clust, 'gray') for clust in all_unique}
    print("Using pre-defined colors from adata.uns")
else:
    # Fallback: generate palette
    palette = sns.color_palette('tab20', len(all_unique))
    cluster_colors = dict(zip(all_unique, palette))
    print("Using generated color palette")

# Coordinates (filtered to exclude uncertain)
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm['spatial'][mask_known]
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = adata.obs['CenterX_global_px'][mask_known]
    y_spatial = adata.obs['CenterY_global_px'][mask_known]

umap_coords = adata.obsm['X_umap'][mask_known]

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial plot
sns.scatterplot(x=x_spatial, y=y_spatial,
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=3, edgecolor=None, ax=axes[0], legend=False)

axes[0].set_title('Adjusted L3 Clusters - Spatial (uncertain excluded)')
axes[0].set_xlabel('X (px)')
axes[0].set_ylabel('Y (px)')
axes[0].invert_yaxis()
axes[0].set_aspect('equal')

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=2, edgecolor=None, ax=axes[1], legend=False)

axes[1].set_title('Adjusted L3 Clusters - UMAP (uncertain excluded)')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

# Shared external legend
handles = [Line2D([0], [0], marker='o', color='w',
                  markerfacecolor=cluster_colors[clust],
                  markersize=10) for clust in all_unique]

fig.legend(handles=handles, labels=all_unique,
           title='Adjusted L3 Clusters',
           bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=10)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

#2T-->bgs4_reseg ('harmony_failed/uncertain' cluster hidden)

In [ ]:
import pandas as pd

label_col = 'adjusted_L3_transferred'

# Ensure categories are ordered as Scanpy sees them
categories = bgs4_lge_mge.obs[label_col].cat.categories

# Scanpy stores colors here
colors = bgs4_lge_mge.uns[f"{label_col}_colors"]

# Build table
color_df = pd.DataFrame({
    "cell_type": categories,
    "scanpy_hex_color": colors
})

print(color_df)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Single heatmap: marker gene expression across cell types (bgs4)
# - Title: "2T  GW24 (bgs4)"
# - Z-score clipped to [-2, 2]
# - Cell counts shown on the right
# - Fully customizable: genes + cell type list

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',
    'DRD2',

    'BACH2',
    'EPHA4',
    'PENK',
    'GAD1',

    'PAX6',
    'OLIG1',

    'OLIG2',
    'SOX10',
    'PDGFRA',
    'MBP',

    'ALDH1L1',
    'GFAP',
    'AQP4',

    'ESAM',
    'FN1',
    'LUM',
    'DCN',

    'P2RY12',
    'CX3CR1',
    'NKX2-1',
]

# 2. Explicitly define cell types to KEEP (bgs4 only)
cell_types_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    '?',

    'GPC',
    'OPC',
    'Astro',

    'EC',
    'PC',
    'VLMC',
    'MGC-1',
]

# 3. Filtering and area-based QC (optional, currently off)
label_col = 'adjusted_L3_transferred'
min_area_um2 = 10
um_per_pixel = 0.12028

bgs4_f = bgs4.copy()

# Uncomment if you want area filtering
# nuc_area = bgs4_f.obs['NucArea'] * (um_per_pixel ** 2)
# cell_area = bgs4_f.obs['Area'] * (um_per_pixel ** 2)
# mask = (nuc_area >= min_area_um2) & (cell_area >= min_area_um2)
# bgs4_f = bgs4_f[mask].copy()

# 4. Restrict to desired cell types
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()

# 5. Pseudobulk z-score function
def pseudobulk_zscore_fine(adata, genes, groupby=label_col):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"bgs4 - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values

    pb = df.groupby(groupby).mean()

    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0).clip(lower=-2, upper=2)

    return pb_z, genes_use

# 6. Compute pseudobulk
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order)
pb4 = pb4.reindex(cell_types_bgs4)

# 7. Cell counts
counts4 = (
    bgs4_f.obs[label_col]
    .value_counts()
    .reindex(pb4.index)
    .fillna(0)
    .astype(int)
)

# 8. Plotting
fig, ax = plt.subplots(
    figsize=(14, len(pb4) * 0.45 + 3)
)

g = sns.heatmap(
    pb4,
    cmap='vlag',
    vmin=-2,
    vmax=2,
    center=0,
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={
        "orientation": "horizontal",
        "pad": 0.25,
        "shrink": 0.6
    },
    ax=ax
)

ax.set_title(
    "2T  GW24 (bgs4) - Selected Cell Types\nPseudobulk marker expression (z-scored, clipped [-2, 2])",
    fontsize=15,
    pad=12
)

ax.set_ylabel("Cell Type", fontsize=13)
ax.set_xlabel("Genes", fontsize=13)
ax.tick_params(axis='x', rotation=90, labelsize=11)

# Colorbar label
cbar = g.collections[0].colorbar
cbar.set_label('z-score', fontsize=11)

# Cell counts on right
for i, n in enumerate(counts4):
    ax.text(
        len(pb4.columns) + 0.25,
        i + 0.5,
        f"n={n:,}",
        va='center',
        ha='left',
        fontsize=9
    )

ax.axvline(len(pb4.columns), color='black', linewidth=1.2, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Single heatmap: marker gene expression across cell types (bgs4)
# - Title: "2T  GW24 (bgs4)"
# - Z-score clipped to [-2, 2]
# - Cell counts shown on the right
# - Fully customizable: genes + cell type list

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',
    'DRD2',

    'BACH2',
    'EPHA4',
    'PENK',
    'GAD1',

    'PAX6',
    'OLIG1',

    'OLIG2',
    'SOX10',
    'PDGFRA',
    'MBP',

    'ALDH1L1',
    'GFAP',
    'AQP4',

    'ESAM',
    'FN1',
    'LUM',
    'DCN',

    'P2RY12',
    'CX3CR1',
    'NKX2-1',
    'MKI67',
    'TOP2A',
    'ASCL1'
]

# 2. Explicitly define cell types to KEEP (bgs4 only)
cell_types_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    #'eMGE',
    '?',
    'GPC',
    'OPC',
    'Astro',

    'EC',
    'PC',
    'VLMC',
    'MGC-1',
]

# 3. Filtering and area-based QC (optional, currently off)
label_col = 'adjusted_L3_transferred'
min_area_um2 = 10
um_per_pixel = 0.12028

bgs4_f = bgs4_lge_mge.copy()

# Uncomment if you want area filtering
# nuc_area = bgs4_f.obs['NucArea'] * (um_per_pixel ** 2)
# cell_area = bgs4_f.obs['Area'] * (um_per_pixel ** 2)
# mask = (nuc_area >= min_area_um2) & (cell_area >= min_area_um2)
# bgs4_f = bgs4_f[mask].copy()

# 4. Restrict to desired cell types
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()

# 5. Pseudobulk z-score function
def pseudobulk_zscore_fine(adata, genes, groupby=label_col):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"bgs4 - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values

    pb = df.groupby(groupby).mean()

    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0).clip(lower=-2, upper=2)

    return pb_z, genes_use

# 6. Compute pseudobulk
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order)
pb4 = pb4.reindex(cell_types_bgs4)

# 7. Cell counts
counts4 = (
    bgs4_f.obs[label_col]
    .value_counts()
    .reindex(pb4.index)
    .fillna(0)
    .astype(int)
)

# 8. Plotting
fig, ax = plt.subplots(
    figsize=(14, len(pb4) * 0.45 + 3)
)

g = sns.heatmap(
    pb4,
    cmap='vlag',
    vmin=-2,
    vmax=2,
    center=0,
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={
        "orientation": "horizontal",
        "pad": 0.25,
        "shrink": 0.6
    },
    ax=ax
)

ax.set_title(
    "2T  GW24 (bgs4) - Selected Cell Types\nPseudobulk marker expression (z-scored, clipped [-2, 2])",
    fontsize=15,
    pad=12
)

ax.set_ylabel("Cell Type", fontsize=13)
ax.set_xlabel("Genes", fontsize=13)
ax.tick_params(axis='x', rotation=90, labelsize=11)

# Colorbar label
cbar = g.collections[0].colorbar
cbar.set_label('z-score', fontsize=11)

# Cell counts on right
for i, n in enumerate(counts4):
    ax.text(
        len(pb4.columns) + 0.25,
        i + 0.5,
        f"n={n:,}",
        va='center',
        ha='left',
        fontsize=9
    )

ax.axvline(len(pb4.columns), color='black', linewidth=1.2, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
label_col = 'adjusted_L3_transferred'

bgs4_lge_mge.obs[label_col] = (
    bgs4_lge_mge.obs[label_col]
    .replace({'?': 'eMGE'})
)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Single heatmap: marker gene expression across cell types (bgs4)
# - Title: "2T  GW24 (bgs4)"
# - Z-score clipped to [-2, 2]
# - Cell counts shown on the right
# - Fully customizable: genes + cell type list

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',
    'DRD2',

    'BACH2',
    'EPHA4',
    'PENK',
    'GAD1',

    'PAX6',
    'OLIG1',

    'OLIG2',
    'SOX10',
    'PDGFRA',
    'MBP',

    'ALDH1L1',
    'GFAP',
    'AQP4',

    'ESAM',
    'FN1',
    'LUM',
    'DCN',

    'P2RY12',
    'CX3CR1',
    'NKX2-1',
    'MKI67',
    'TOP2A',
    'ASCL1'
]

# 2. Explicitly define cell types to KEEP (bgs4 only)
cell_types_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    #'?',
    'GPC',
    'OPC',
    'Astro',

    'EC',
    'PC',
    #'VLMC',
    'MGC-1',
]

# 3. Filtering and area-based QC (optional, currently off)
label_col = 'adjusted_L3_transferred'
min_area_um2 = 10
um_per_pixel = 0.12028

bgs4_f = bgs4_lge_mge.copy()

# Uncomment if you want area filtering
# nuc_area = bgs4_f.obs['NucArea'] * (um_per_pixel ** 2)
# cell_area = bgs4_f.obs['Area'] * (um_per_pixel ** 2)
# mask = (nuc_area >= min_area_um2) & (cell_area >= min_area_um2)
# bgs4_f = bgs4_f[mask].copy()

# 4. Restrict to desired cell types
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()

# 5. Pseudobulk z-score function
def pseudobulk_zscore_fine(adata, genes, groupby=label_col):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"bgs4 - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values

    pb = df.groupby(groupby).mean()

    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0).clip(lower=-2, upper=2)

    return pb_z, genes_use

# 6. Compute pseudobulk
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order)
pb4 = pb4.reindex(cell_types_bgs4)

# 7. Cell counts
counts4 = (
    bgs4_f.obs[label_col]
    .value_counts()
    .reindex(pb4.index)
    .fillna(0)
    .astype(int)
)

# 8. Plotting
fig, ax = plt.subplots(
    figsize=(14, len(pb4) * 0.45 + 3)
)

g = sns.heatmap(
    pb4,
    cmap='vlag',
    vmin=-2,
    vmax=2,
    center=0,
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={
        "orientation": "horizontal",
        "pad": 0.25,
        "shrink": 0.6
    },
    ax=ax
)

ax.set_title(
    "2T  GW24 (bgs4) - Selected Cell Types\nPseudobulk marker expression (z-scored, clipped [-2, 2])",
    fontsize=15,
    pad=12
)

ax.set_ylabel("Cell Type", fontsize=13)
ax.set_xlabel("Genes", fontsize=13)
ax.tick_params(axis='x', rotation=90, labelsize=11)

# Colorbar label
cbar = g.collections[0].colorbar
cbar.set_label('z-score', fontsize=11)

# Cell counts on right
for i, n in enumerate(counts4):
    ax.text(
        len(pb4.columns) + 0.25,
        i + 0.5,
        f"n={n:,}",
        va='center',
        ha='left',
        fontsize=9
    )

ax.axvline(len(pb4.columns), color='black', linewidth=1.2, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# Matplotlib settings for Illustrator-friendly SVG
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'   # keep text editable in SVG

# Paths
outdir = Path("20260114 - figs")
outdir.mkdir(exist_ok=True)

# Data
adata = bgs4_lge_mge
categ = 'adjusted_L3_transferred'

# 1. EXPLICIT cell type order
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# 2. Explicit colors
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# 3. Keep only cell types present
present = set(adata.obs[categ].dropna().unique())
cell_type_order = [ct for ct in cell_type_order if ct in present]
n_clusters = len(cell_type_order)

missing = set(cell_type_order) - set(cell_type_colors)
assert not missing, f"Missing colors for: {missing}"

# 4. Spatial coords + global limits
spatial = adata.obsm['spatial']
xlim = (np.nanmin(spatial[:, 0]), np.nanmax(spatial[:, 0]))
ylim = (np.nanmin(spatial[:, 1]), np.nanmax(spatial[:, 1]))

grey_color = '#D3D3D3'
spot_size = 0.03

# 5. Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten()

# 6. Plot panels
for idx, cell_type in enumerate(cell_type_order):
    ax = axes[idx]
    mask = (adata.obs[categ] == cell_type).to_numpy()

    # Background (all other cells)
    ax.scatter(
        spatial[~mask, 0],
        spatial[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.20,
        rasterized=True
    )

    # Foreground (cell type)
    ax.scatter(
        spatial[mask, 0],
        spatial[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

# 7. Remove empty panels
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.subplots_adjust(
    wspace=-0.75,
    hspace=0.25
)

# 8. Save multi-panel SVG
fig.savefig(
    outdir / "bgs4_celltype_panels.svg",
    bbox_inches="tight",
    dpi=600
)

# 9. Save one SVG per cell type
for cell_type in cell_type_order:
    fig_single, ax = plt.subplots(figsize=(6, 6))
    mask = (adata.obs[categ] == cell_type).to_numpy()

    ax.scatter(
        spatial[~mask, 0],
        spatial[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.20,
        rasterized=True
    )
    ax.scatter(
        spatial[mask, 0],
        spatial[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

    fig_single.savefig(
        outdir / f"bgs4_{cell_type}.svg",
        bbox_inches="tight",
        dpi=600
    )
    plt.close(fig_single)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# Matplotlib settings for Illustrator-friendly PDF
# - editable text (font kept as font)
# - rasterized points (small file size)
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',  # harmless to keep
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# Paths
outdir = Path("./20260115 - pdf figs/bgs4")
outdir.mkdir(parents=True, exist_ok=True)

# Data
#   adata     = filtered + typed (foreground)
#   adata_bg  = full object (background only; includes low-UMI cells)
adata = bgs4_lge_mge
adata_bg = bgs4_lge_mge              # <-- change this if your full object has a different name
categ = 'adjusted_L3_transferred'

# 1. EXPLICIT cell type order
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# 2. Explicit colors
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# 3. Keep only cell types present in the typed object
present = set(adata.obs[categ].dropna().unique())
cell_type_order = [ct for ct in cell_type_order if ct in present]
n_clusters = len(cell_type_order)

missing = set(cell_type_order) - set(cell_type_colors)
assert not missing, f"Missing colors for: {missing}"

# 4. Spatial coords + global limits (from FULL dataset)
spatial_bg = adata_bg.obsm['spatial']
spatial_fg = adata.obsm['spatial']

xlim = (np.nanmin(spatial_bg[:, 0]), np.nanmax(spatial_bg[:, 0]))
ylim = (np.nanmin(spatial_bg[:, 1]), np.nanmax(spatial_bg[:, 1]))

grey_color = '#D3D3D3'
spot_size_bg = 0.002   # background dots (small)
spot_size_fg = 0.10   # foreground dots (bigger)

# 5. Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten()

# 6. Plot panels (background from full object; foreground from typed object)
for idx, cell_type in enumerate(cell_type_order):
    ax = axes[idx]
    mask = (adata.obs[categ] == cell_type).to_numpy()

    # Background: ALL cells from bgs4_full in grey (includes low-UMI cells)
    ax.scatter(
        spatial_bg[:, 0],
        spatial_bg[:, 1],
        c=grey_color,
        s=spot_size_bg,
        alpha=1.0,
        rasterized=True
    )

    # Foreground: typed cells from filtered object in color
    ax.scatter(
        spatial_fg[mask, 0],
        spatial_fg[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size_fg,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

# 7. Remove empty panels
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.subplots_adjust(
    wspace=-0.75,
    hspace=0.25
)

# 8. Save multi-panel PDF
fig.savefig(
    outdir / "bgs4_celltype_panels.pdf",
    format="pdf",
    bbox_inches="tight",
    dpi=300,
    transparent=True
)

# 9. Save one PDF per cell type
for cell_type in cell_type_order:
    fig_single, ax = plt.subplots(figsize=(6, 6))
    mask = (adata.obs[categ] == cell_type).to_numpy()

    ax.scatter(
        spatial_bg[:, 0], spatial_bg[:, 1],
        c=grey_color, s=spot_size_bg, alpha=1, rasterized=True
    )
    ax.scatter(
        spatial_fg[mask, 0], spatial_fg[mask, 1],
        c=cell_type_colors[cell_type], s=spot_size_fg, alpha=1.0, rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

    fig_single.savefig(
        outdir / f"bgs4_{cell_type}.pdf",
        format="pdf",
        bbox_inches="tight",
        dpi=300,
        transparent=True
    )
    plt.close(fig_single)

plt.show()
print(f"Saved multi-panel + {len(cell_type_order)} single-panel PDFs to: {outdir}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# Matplotlib settings for Illustrator-friendly PDF
# - editable text (font kept as font)
# - rasterized points (small file size)
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',  # harmless to keep
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# Paths
outdir = Path("./20260115 - pdf figs/bgs4")
outdir.mkdir(parents=True, exist_ok=True)

# Data
#   adata     = filtered + typed (foreground)
#   adata_bg  = full object (background only; includes low-UMI cells)
adata = bgs4_lge_mge
adata_bg = bgs4_lge_mge              # <-- change this if your full object has a different name
categ = 'adjusted_L3_transferred'
uncertain_label = 'uncertain'        # <-- change if needed ('Uncertain', '?', etc.)

# 0. REMOVE uncertain cells from BOTH foreground + background
fg_keep = (adata.obs[categ].notna()) & (adata.obs[categ] != uncertain_label)
bg_keep = (adata_bg.obs[categ].notna()) & (adata_bg.obs[categ] != uncertain_label)

adata_f = adata[fg_keep].copy()
adata_bg_f = adata_bg[bg_keep].copy()

# 1. EXPLICIT cell type order
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# 2. Explicit colors
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# 3. Keep only cell types present in the (uncertain-removed) foreground object
present = set(adata_f.obs[categ].dropna().unique())
cell_type_order = [ct for ct in cell_type_order if ct in present]
n_clusters = len(cell_type_order)

missing = set(cell_type_order) - set(cell_type_colors)
assert not missing, f"Missing colors for: {missing}"

# 4. Spatial coords + global limits (from uncertain-removed background dataset)
spatial_bg = adata_bg_f.obsm['spatial']
spatial_fg = adata_f.obsm['spatial']

xlim = (np.nanmin(spatial_bg[:, 0]), np.nanmax(spatial_bg[:, 0]))
ylim = (np.nanmin(spatial_bg[:, 1]), np.nanmax(spatial_bg[:, 1]))

grey_color = '#D3D3D3'
spot_size_bg = 0.002   # background dots (small)
spot_size_fg = 0.10    # foreground dots (bigger)

# 5. Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten()

# 6. Plot panels (background from full object; foreground from typed object)
#    both have "uncertain" removed
for idx, cell_type in enumerate(cell_type_order):
    ax = axes[idx]
    mask = (adata_f.obs[categ] == cell_type).to_numpy()

    # Background: ALL non-uncertain cells in grey
    ax.scatter(
        spatial_bg[:, 0],
        spatial_bg[:, 1],
        c=grey_color,
        s=spot_size_bg,
        alpha=1.0,
        rasterized=True
    )

    # Foreground: this cell type in color (non-uncertain only)
    ax.scatter(
        spatial_fg[mask, 0],
        spatial_fg[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size_fg,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

# 7. Remove empty panels
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.subplots_adjust(
    wspace=-0.75,
    hspace=0.25
)

# 8. Save multi-panel PDF
fig.savefig(
    outdir / "bgs4_celltype_panels_no_uncertain.pdf",
    format="pdf",
    bbox_inches="tight",
    dpi=300,
    transparent=True
)

# 9. Save one PDF per cell type
for cell_type in cell_type_order:
    fig_single, ax = plt.subplots(figsize=(6, 6))
    mask = (adata_f.obs[categ] == cell_type).to_numpy()

    ax.scatter(
        spatial_bg[:, 0],
        spatial_bg[:, 1],
        c=grey_color,
        s=spot_size_bg,
        alpha=1.0,
        rasterized=True
    )
    ax.scatter(
        spatial_fg[mask, 0],
        spatial_fg[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size_fg,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

    fig_single.savefig(
        outdir / f"bgs4_{cell_type}_no_uncertain.pdf",
        format="pdf",
        bbox_inches="tight",
        dpi=300,
        transparent=True
    )
    plt.close(fig_single)

plt.show()
print(f"Saved multi-panel + {len(cell_type_order)} single-panel PDFs to: {outdir}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Spatial gene expression panels (CosMx)
# - Version A: "normal" expression (log1p(CP10K)) per cell
# - Version B: z-scored expression across cells (computed on same normal values)
# - Uncertain cells shown as GREY background to preserve tissue structure
# - No saving; just plotting

# Inputs
adata = bgs4_lge_mge
categ = 'adjusted_L3_transferred'
uncertain_label = 'uncertain'   # change if needed ('Uncertain', '?', etc.)

genes = [
    'DRD1', 'DRD2',
    'BACH2', 'EPHA4', 'PENK', 'GAD1', 'NKX2-1', 'EGFR', 'PAX6',
    'OLIG1', 'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4',
    'ESAM', 'FN1', 'DCN',
    'P2RY12', 'CX3CR1'
]

# Helpers
def _get_dense_vector(adata, gene, prefer_layer='counts_downsampled'):
    """Return dense 1D array of expression for one gene."""
    if gene not in adata.var_names:
        return None

    if prefer_layer is not None and (hasattr(adata, "layers")) and (prefer_layer in adata.layers):
        x = adata[:, gene].layers[prefer_layer]
    else:
        x = adata[:, gene].X

    if sp.issparse(x):
        x = x.toarray()
    x = np.asarray(x).reshape(-1)
    return x

def _log1p_cp10k(counts_vec, libsize):
    """
    "Normal CosMx-ish" view:
    log1p(counts per 10k) using per-cell library size (from counts layer if available, else from X).
    """
    denom = np.maximum(libsize, 1e-8)
    cp10k = counts_vec / denom * 1e4
    return np.log1p(cp10k)

def _compute_libsize(adata, prefer_layer='counts_downsampled'):
    """Per-cell library size based on prefer_layer if present, else X."""
    if prefer_layer is not None and (hasattr(adata, "layers")) and (prefer_layer in adata.layers):
        X = adata.layers[prefer_layer]
    else:
        X = adata.X

    if sp.issparse(X):
        lib = np.asarray(X.sum(axis=1)).reshape(-1)
    else:
        lib = np.asarray(X.sum(axis=1)).reshape(-1)
    return lib

def _plot_gene_panels(
    adata,
    genes,
    value_mode="normal",  # "normal" or "zscore"
    prefer_layer='counts_downsampled',
    ncols=6,
    figsize_per_panel=3.2,
    bg_size=0.002,
    fg_size=0.10,
    bg_alpha=1.0,
    fg_alpha=1.0,
):
    """
    Plot spatial panels for a list of genes.
    Background: uncertain cells in grey (plus any non-uncertain cells in very light grey if you want tissue context).
    Foreground: non-uncertain cells colored by gene value.
    """
    spatial = adata.obsm['spatial']
    xlim = (np.nanmin(spatial[:, 0]), np.nanmax(spatial[:, 0]))
    ylim = (np.nanmin(spatial[:, 1]), np.nanmax(spatial[:, 1]))

    # define masks
    labels = adata.obs[categ].astype(str).to_numpy()
    is_uncertain = (labels == uncertain_label)
    is_typed = ~is_uncertain

    # library size for "normal" mode
    libsize = _compute_libsize(adata, prefer_layer=prefer_layer)

    # keep only genes present
    genes_present = [g for g in genes if g in adata.var_names]
    genes_missing = [g for g in genes if g not in adata.var_names]
    if len(genes_missing) > 0:
        print(f"Skipping missing genes ({len(genes_missing)}): {genes_missing}")

    n = len(genes_present)
    if n == 0:
        raise ValueError("None of the requested genes are in adata.var_names")

    nrows = int(np.ceil(n / ncols))
    fig = plt.figure(figsize=(figsize_per_panel * ncols, figsize_per_panel * nrows))
    axes = fig.subplots(nrows, ncols).flatten()

    # compute values + global limits for consistent color scaling
    vals = {}
    for g in genes_present:
        v_raw = _get_dense_vector(adata, g, prefer_layer=prefer_layer)
        if v_raw is None:
            continue

        # "normal" values: log1p(CP10K) from counts-like layer if possible
        v_norm = _log1p_cp10k(v_raw, libsize)

        if value_mode == "normal":
            v = v_norm
        elif value_mode == "zscore":
            # z-score across typed cells only (prevents uncertain from affecting mean/var)
            mu = np.nanmean(v_norm[is_typed])
            sd = np.nanstd(v_norm[is_typed]) + 1e-8
            v = (v_norm - mu) / sd
        else:
            raise ValueError("value_mode must be 'normal' or 'zscore'")

        vals[g] = v

    # global scale computed on typed cells only (so uncertain doesn't distort scaling)
    all_v = np.concatenate([vals[g][is_typed] for g in genes_present])
    vmin = np.nanpercentile(all_v, 1)
    vmax = np.nanpercentile(all_v, 99)
    if value_mode == "zscore":
        # symmetric looks nicer for z
        absmax = max(abs(vmin), abs(vmax))
        vmin, vmax = -absmax, absmax

    for i, g in enumerate(genes_present):
        ax = axes[i]

        # Background: uncertain cells in grey to show tissue structure
        ax.scatter(
            spatial[is_uncertain, 0],
            spatial[is_uncertain, 1],
            c="#D3D3D3",
            s=bg_size,
            alpha=bg_alpha,
            rasterized=True
        )

        # Foreground: typed cells colored by gene value
        sc = ax.scatter(
            spatial[is_typed, 0],
            spatial[is_typed, 1],
            c=vals[g][is_typed],
            s=fg_size,
            alpha=fg_alpha,
            rasterized=True,
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(g, fontsize=14)
        ax.set_aspect('equal')
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.invert_yaxis()
        ax.axis('off')

    # remove empty axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    # one shared colorbar
    cbar = fig.colorbar(sc, ax=axes[:n], fraction=0.02, pad=0.01)
    if value_mode == "normal":
        cbar.set_label("log1p(CP10K)", fontsize=12)
        fig.suptitle("Spatial gene expression (normal): log1p(CP10K)\nUncertain cells shown in grey background",
                     fontsize=16, y=0.995)
    else:
        cbar.set_label("z-score of log1p(CP10K) across typed cells", fontsize=12)
        fig.suptitle("Spatial gene expression (z-scored): z(log1p(CP10K)) across typed cells\nUncertain cells shown in grey background",
                     fontsize=16, y=0.995)

    plt.subplots_adjust(wspace=0.02, hspace=0.20, top=0.92)
    plt.show()

# VERSION A: "normal" CosMx-style view (log1p(CP10K))
_plot_gene_panels(
    adata=adata,
    genes=genes,
    value_mode="normal",
    prefer_layer="counts_downsampled",  # will fall back to adata.X if absent
    ncols=6,
    figsize_per_panel=3.2,
    bg_size=0.002,
    fg_size=0.10,
    bg_alpha=1.0,
    fg_alpha=1.0,
)

# VERSION B: z-scored view (z-score of the same normal values)
_plot_gene_panels(
    adata=adata,
    genes=genes,
    value_mode="zscore",
    prefer_layer="counts_downsampled",  # will fall back to adata.X if absent
    ncols=6,
    figsize_per_panel=3.2,
    bg_size=0.002,
    fg_size=0.10,
    bg_alpha=1.0,
    fg_alpha=1.0,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Spatial gene expression panels (CosMx)
# - Version A: "normal" expression (log1p(CP10K)) per cell
# - Version B: z-scored expression across cells (computed on same normal values)
# - Uncertain cells shown as GREY background to preserve tissue structure
# UPDATES YOU REQUESTED:
# 1) Lower-end threshold cutoff (raise vmin so low values don't dominate/purple)
# 2) Smaller dots
# 3) Less whitespace between panels (bigger plots)

# Inputs
adata = bgs4_lge_mge
categ = 'adjusted_L3_transferred'
uncertain_label = 'uncertain'   # change if needed ('Uncertain', '?', etc.)

genes = [
    'DRD1', 'DRD2',
    'BACH2', 'EPHA4', 'PENK', 'GAD1', 'NKX2-1', 'EGFR', 'PAX6',
    'OLIG1', 'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4',
    'ESAM', 'FN1', 'DCN',
    'P2RY12', 'CX3CR1'
]

# Display controls (tune these)
# Smaller dots
spot_size_bg = 0.0008
spot_size_fg = 0.03

# Less whitespace (bigger panels)
panel_ncols = 6
panel_figsize_per = 3.0
wspace = 0.005
hspace = 0.08
top_margin = 0.90

# Raise the lower cutoff to reduce "too much purple"
# - normal: use higher percentile as vmin
# - zscore: clip lower tail more aggressively
normal_vmin_percentile = 30   # was ~1 before; raising reduces purple low background
normal_vmax_percentile = 99

zscore_vmin_percentile = 10   # for z-score, still percentile-based but symmetric after
zscore_vmax_percentile = 99

# Optional: hard floor cutoff (applies AFTER percentile scaling)
# Set to None to disable
hard_floor_normal = None      # e.g. 0.15
hard_floor_zscore = None      # e.g. -0.25

# Helpers
def _get_dense_vector(adata, gene, prefer_layer='counts_downsampled'):
    """Return dense 1D array of expression for one gene."""
    if gene not in adata.var_names:
        return None

    if prefer_layer is not None and (hasattr(adata, "layers")) and (prefer_layer in adata.layers):
        x = adata[:, gene].layers[prefer_layer]
    else:
        x = adata[:, gene].X

    if sp.issparse(x):
        x = x.toarray()
    x = np.asarray(x).reshape(-1)
    return x

def _compute_libsize(adata, prefer_layer='counts_downsampled'):
    """Per-cell library size based on prefer_layer if present, else X."""
    if prefer_layer is not None and (hasattr(adata, "layers")) and (prefer_layer in adata.layers):
        X = adata.layers[prefer_layer]
    else:
        X = adata.X

    if sp.issparse(X):
        lib = np.asarray(X.sum(axis=1)).reshape(-1)
    else:
        lib = np.asarray(X.sum(axis=1)).reshape(-1)
    return lib

def _log1p_cp10k(counts_vec, libsize):
    """log1p(counts per 10k) using per-cell library size."""
    denom = np.maximum(libsize, 1e-8)
    cp10k = counts_vec / denom * 1e4
    return np.log1p(cp10k)

def _plot_gene_panels(
    adata,
    genes,
    value_mode="normal",  # "normal" or "zscore"
    prefer_layer='counts_downsampled',
    ncols=6,
    figsize_per_panel=3.0,
    bg_size=0.001,
    fg_size=0.03,
    bg_alpha=1.0,
    fg_alpha=1.0,
    vmin_pct=15,
    vmax_pct=99,
    hard_floor=None,
    wspace=0.005,
    hspace=0.08,
    top=0.90,
):
    spatial = adata.obsm['spatial']
    xlim = (np.nanmin(spatial[:, 0]), np.nanmax(spatial[:, 0]))
    ylim = (np.nanmin(spatial[:, 1]), np.nanmax(spatial[:, 1]))

    labels = adata.obs[categ].astype(str).to_numpy()
    is_uncertain = (labels == uncertain_label)
    is_typed = ~is_uncertain

    libsize = _compute_libsize(adata, prefer_layer=prefer_layer)

    genes_present = [g for g in genes if g in adata.var_names]
    genes_missing = [g for g in genes if g not in adata.var_names]
    if len(genes_missing) > 0:
        print(f"Skipping missing genes ({len(genes_missing)}): {genes_missing}")

    n = len(genes_present)
    if n == 0:
        raise ValueError("None of the requested genes are in adata.var_names")

    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_panel * ncols, figsize_per_panel * nrows),
        squeeze=False
    )
    axes = axes.flatten()

    # compute per-gene values
    vals = {}
    for g in genes_present:
        v_raw = _get_dense_vector(adata, g, prefer_layer=prefer_layer)
        v_norm = _log1p_cp10k(v_raw, libsize)

        if value_mode == "normal":
            v = v_norm
        elif value_mode == "zscore":
            mu = np.nanmean(v_norm[is_typed])
            sd = np.nanstd(v_norm[is_typed]) + 1e-8
            v = (v_norm - mu) / sd
        else:
            raise ValueError("value_mode must be 'normal' or 'zscore'")

        vals[g] = v

    # global color scaling (typed cells only)
    all_v = np.concatenate([vals[g][is_typed] for g in genes_present])
    vmin = np.nanpercentile(all_v, vmin_pct)
    vmax = np.nanpercentile(all_v, vmax_pct)

    # optional hard floor to kill low-end purple even more
    if hard_floor is not None:
        vmin = max(vmin, hard_floor)

    # symmetric for z-score (still respects raised vmin via percentile)
    if value_mode == "zscore":
        absmax = max(abs(vmin), abs(vmax))
        vmin, vmax = -absmax, absmax
        if hard_floor is not None:
            # if user sets hard_floor for zscore, treat it as lower bound on vmin magnitude
            absmax = max(abs(hard_floor), absmax)
            vmin, vmax = -absmax, absmax

    # plot
    last_sc = None
    for i, g in enumerate(genes_present):
        ax = axes[i]

        # Background: uncertain cells only
        ax.scatter(
            spatial[is_uncertain, 0],
            spatial[is_uncertain, 1],
            c="#D3D3D3",
            s=bg_size,
            alpha=bg_alpha,
            rasterized=True
        )

        # Foreground: typed cells colored by expression
        last_sc = ax.scatter(
            spatial[is_typed, 0],
            spatial[is_typed, 1],
            c=vals[g][is_typed],
            s=fg_size,
            alpha=fg_alpha,
            rasterized=True,
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(g, fontsize=13, pad=2)
        ax.set_aspect('equal')
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.invert_yaxis()
        ax.axis('off')

    # remove empty axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    # shared colorbar
    cbar = fig.colorbar(last_sc, ax=axes[:n], fraction=0.02, pad=0.005)
    if value_mode == "normal":
        cbar.set_label("log1p(CP10K)", fontsize=12)
        fig.suptitle(
            "Spatial gene expression (normal): log1p(CP10K)\nUncertain cells in grey background",
            fontsize=16,
            y=0.99
        )
    else:
        cbar.set_label("z-score of log1p(CP10K) across typed cells", fontsize=12)
        fig.suptitle(
            "Spatial gene expression (z-scored): z(log1p(CP10K)) across typed cells\nUncertain cells in grey background",
            fontsize=16,
            y=0.99
        )

    # tighten whitespace
    plt.subplots_adjust(wspace=wspace, hspace=hspace, top=top)
    plt.show()

# VERSION A: "normal" (log1p(CP10K)) with raised lower cutoff
_plot_gene_panels(
    adata=adata,
    genes=genes,
    value_mode="normal",
    prefer_layer="counts_downsampled",
    ncols=panel_ncols,
    figsize_per_panel=panel_figsize_per,
    bg_size=spot_size_bg,
    fg_size=spot_size_fg,
    vmin_pct=normal_vmin_percentile,
    vmax_pct=normal_vmax_percentile,
    hard_floor=hard_floor_normal,
    wspace=wspace,
    hspace=hspace,
    top=top_margin,
)

# VERSION B: z-scored with raised lower cutoff
_plot_gene_panels(
    adata=adata,
    genes=genes,
    value_mode="zscore",
    prefer_layer="counts_downsampled",
    ncols=panel_ncols,
    figsize_per_panel=panel_figsize_per,
    bg_size=spot_size_bg,
    fg_size=spot_size_fg,
    vmin_pct=zscore_vmin_percentile,
    vmax_pct=zscore_vmax_percentile,
    hard_floor=hard_floor_zscore,
    wspace=wspace,
    hspace=hspace,
    top=top_margin,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Spatial gene expression panels (CosMx)
# - Version A: "normal" expression (log1p(CP10K)) per cell
# - Version B: z-scored expression across cells (computed on same normal values)
# UPDATES YOU REQUESTED:
# 1) INCLUDE uncertain cells in the expression coloring (no grey background)
# 2) Raise lower cutoff: vmin = 30th percentile (typed+uncertain included)
# 3) Smaller dots + tighter whitespace (kept from prior)
# - No saving; just plotting

# Inputs
adata = bgs4_lge_mge
genes = [
    'DRD1', 'DRD2',
    'BACH2', 'EPHA4', 'PENK', 'GAD1', 'NKX2-1', 'EGFR', 'PAX6',
    'OLIG1', 'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4',
    'ESAM', 'FN1', 'DCN',
    'P2RY12', 'CX3CR1'
]

# Display controls
spot_size = 0.02         # smaller dots
panel_ncols = 6
panel_figsize_per = 3.0  # per-panel scale (overall figure scales with rows/cols)
wspace = 0.005
hspace = 0.08
top_margin = 0.90

# Raise lower cutoff to reduce "too much purple"
VMIN_PERCENTILE = 30
VMAX_PERCENTILE = 99

# Helpers
def _get_dense_vector(adata, gene, prefer_layer='counts_downsampled'):
    if gene not in adata.var_names:
        return None

    if prefer_layer is not None and hasattr(adata, "layers") and (prefer_layer in adata.layers):
        x = adata[:, gene].layers[prefer_layer]
    else:
        x = adata[:, gene].X

    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).reshape(-1)

def _compute_libsize(adata, prefer_layer='counts_downsampled'):
    if prefer_layer is not None and hasattr(adata, "layers") and (prefer_layer in adata.layers):
        X = adata.layers[prefer_layer]
    else:
        X = adata.X

    if sp.issparse(X):
        return np.asarray(X.sum(axis=1)).reshape(-1)
    return np.asarray(X.sum(axis=1)).reshape(-1)

def _log1p_cp10k(counts_vec, libsize):
    denom = np.maximum(libsize, 1e-8)
    cp10k = counts_vec / denom * 1e4
    return np.log1p(cp10k)

def _plot_gene_panels(
    adata,
    genes,
    value_mode="normal",  # "normal" or "zscore"
    prefer_layer='counts_downsampled',
    ncols=6,
    figsize_per_panel=3.0,
    spot_size=0.02,
    vmin_pct=30,
    vmax_pct=99,
    wspace=0.005,
    hspace=0.08,
    top=0.90,
):
    spatial = adata.obsm['spatial']
    xlim = (np.nanmin(spatial[:, 0]), np.nanmax(spatial[:, 0]))
    ylim = (np.nanmin(spatial[:, 1]), np.nanmax(spatial[:, 1]))

    genes_present = [g for g in genes if g in adata.var_names]
    genes_missing = [g for g in genes if g not in adata.var_names]
    if genes_missing:
        print(f"Skipping missing genes ({len(genes_missing)}): {genes_missing}")

    n = len(genes_present)
    if n == 0:
        raise ValueError("None of the requested genes are in adata.var_names")

    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_panel * ncols, figsize_per_panel * nrows),
        squeeze=False
    )
    axes = axes.flatten()

    libsize = _compute_libsize(adata, prefer_layer=prefer_layer)

    # compute values
    vals = {}
    for g in genes_present:
        v_raw = _get_dense_vector(adata, g, prefer_layer=prefer_layer)
        v_norm = _log1p_cp10k(v_raw, libsize)

        if value_mode == "normal":
            v = v_norm
        elif value_mode == "zscore":
            mu = np.nanmean(v_norm)
            sd = np.nanstd(v_norm) + 1e-8
            v = (v_norm - mu) / sd
        else:
            raise ValueError("value_mode must be 'normal' or 'zscore'")

        vals[g] = v

    # global color scaling (ALL cells, including uncertain)
    all_v = np.concatenate([vals[g] for g in genes_present])
    vmin = np.nanpercentile(all_v, vmin_pct)
    vmax = np.nanpercentile(all_v, vmax_pct)

    # symmetric for z-score (but still uses percentile to set magnitude)
    if value_mode == "zscore":
        absmax = max(abs(vmin), abs(vmax))
        vmin, vmax = -absmax, absmax

    last_sc = None
    for i, g in enumerate(genes_present):
        ax = axes[i]

        last_sc = ax.scatter(
            spatial[:, 0],
            spatial[:, 1],
            c=vals[g],
            s=spot_size,
            alpha=1.0,
            rasterized=True,
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(g, fontsize=13, pad=2)
        ax.set_aspect('equal')
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.invert_yaxis()
        ax.axis('off')

    # remove empty axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    # shared colorbar
    cbar = fig.colorbar(last_sc, ax=axes[:n], fraction=0.02, pad=0.005)
    if value_mode == "normal":
        cbar.set_label(f"log1p(CP10K)  (vmin=p{vmin_pct}, vmax=p{vmax_pct})", fontsize=12)
        fig.suptitle(
            "Spatial gene expression (normal): log1p(CP10K)\nAll cells included (typed + uncertain)",
            fontsize=16,
            y=0.99
        )
    else:
        cbar.set_label(f"z-score(log1p(CP10K))  (|v|≈p{vmax_pct}, vmin=p{vmin_pct})", fontsize=12)
        fig.suptitle(
            "Spatial gene expression (z-scored): z(log1p(CP10K))\nAll cells included (typed + uncertain)",
            fontsize=16,
            y=0.99
        )

    plt.subplots_adjust(wspace=wspace, hspace=hspace, top=top)
    plt.show()

# VERSION A: "normal" (log1p(CP10K)), vmin = 30th percentile
_plot_gene_panels(
    adata=adata,
    genes=genes,
    value_mode="normal",
    prefer_layer="counts_downsampled",
    ncols=panel_ncols,
    figsize_per_panel=panel_figsize_per,
    spot_size=spot_size,
    vmin_pct=VMIN_PERCENTILE,
    vmax_pct=VMAX_PERCENTILE,
    wspace=wspace,
    hspace=hspace,
    top=top_margin,
)

# VERSION B: z-scored, vmin = 30th percentile (then symmetrized)
_plot_gene_panels(
    adata=adata,
    genes=genes,
    value_mode="zscore",
    prefer_layer="counts_downsampled",
    ncols=panel_ncols,
    figsize_per_panel=panel_figsize_per,
    spot_size=spot_size,
    vmin_pct=VMIN_PERCENTILE,
    vmax_pct=VMAX_PERCENTILE,
    wspace=wspace,
    hspace=hspace,
    top=top_margin,
)

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260108_bgs5_filtered_snm3_3T_L3.h5ad")

#3T --> bgs5_filtered hat was made just now
bgs5

In [ ]:
bgs5_full = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20251130/MBbrain6Kbasalslide5_unfiltered_normalized1.h5ad")

#full object with the low umi cells as well
bgs5_full

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Use your AnnData object
adata = bgs5  # or bgs4_lge_mge if that's the one you're using

# Get cluster annotation
clusters = adata.obs['adjusted_L3_transferred']
cluster_str = clusters.astype(str)

# Exclude 'uncertain' cells entirely
mask_known = cluster_str != 'uncertain'
cluster_str_filtered = cluster_str[mask_known]

# Get unique known clusters and sort nicely (numbers first, then others)
numeric = pd.to_numeric(clusters[mask_known], errors='coerce')
is_numeric = numeric.notna()
numeric_unique = np.sort(numeric[is_numeric].unique().astype(int))
non_numeric_unique = np.sort(clusters[mask_known & ~is_numeric].unique())

all_unique = [str(c) for c in numeric_unique] + [str(c) for c in non_numeric_unique]

# Use pre-defined colors if available (recommended!)
if ('adjusted_L3_transferred' in adata.obs and
    adata.obs['adjusted_L3_transferred'].dtype.name == 'category' and
    'adjusted_L3_transferred_colors' in adata.uns):

    categories = adata.obs['adjusted_L3_transferred'].cat.categories
    known_categories = [cat for cat in categories if str(cat) != 'uncertain']
    known_categories_str = [str(cat) for cat in known_categories]
    color_list = adata.uns['adjusted_L3_transferred_colors']
    # Match colors to known categories
    cat_to_color = dict(zip([str(cat) for cat in categories], color_list))
    cluster_colors = {clust: cat_to_color.get(clust, 'gray') for clust in all_unique}
    print("Using pre-defined colors from adata.uns")
else:
    # Fallback: generate palette
    palette = sns.color_palette('tab20', len(all_unique))
    cluster_colors = dict(zip(all_unique, palette))
    print("Using generated color palette")

# Coordinates (filtered to exclude uncertain)
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm['spatial'][mask_known]
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = adata.obs['CenterX_global_px'][mask_known]
    y_spatial = adata.obs['CenterY_global_px'][mask_known]

umap_coords = adata.obsm['X_umap'][mask_known]

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial plot
sns.scatterplot(x=x_spatial, y=y_spatial,
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=3, edgecolor=None, ax=axes[0], legend=False)

axes[0].set_title('Adjusted L3 Clusters - Spatial (uncertain excluded)')
axes[0].set_xlabel('X (px)')
axes[0].set_ylabel('Y (px)')
axes[0].invert_yaxis()
axes[0].set_aspect('equal')

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=2, edgecolor=None, ax=axes[1], legend=False)

axes[1].set_title('Adjusted L3 Clusters - UMAP (uncertain excluded)')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

# Shared external legend
handles = [Line2D([0], [0], marker='o', color='w',
                  markerfacecolor=cluster_colors[clust],
                  markersize=10) for clust in all_unique]

fig.legend(handles=handles, labels=all_unique,
           title='Adjusted L3 Clusters',
           bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=10)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

#2T-->bgs4_reseg ('harmony_failed/uncertain' cluster hidden)

In [ ]:
import pandas as pd

label_col = 'adjusted_L3_transferred'

# Ensure categories are ordered as Scanpy sees them
categories = bgs5.obs[label_col].cat.categories

# Scanpy stores colors here
colors = bgs5.uns[f"{label_col}_colors"]

# Build table
color_df = pd.DataFrame({
    "cell_type": categories,
    "scanpy_hex_color": colors
})

print(color_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

adata = bgs5
categ = 'adjusted_L3_transferred'

# Get unique clusters
clusters = sorted(adata.obs[categ].unique())
n_clusters = len(clusters)

# Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

# Create figure
fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten() if n_clusters > 1 else [axes]

# Get spatial coordinates
spatial_coords = adata.obsm['spatial']

# Get colors for clusters
if f'{categ}_colors' in adata.uns:
    cluster_colors = dict(zip(adata.obs[categ].cat.categories, adata.uns[f'{categ}_colors']))
else:
    from matplotlib import cm
    cluster_colors = {c: cm.tab20(i % 20) for i, c in enumerate(clusters)}

grey_color = '#D3D3D3'
spot_size = 0.5

# Plot each cluster
for idx, cluster in enumerate(clusters):
    ax = axes[idx]

    # Create mask for this cluster
    mask = adata.obs[categ] == cluster

    # Plot grey background (all other cells)
    ax.scatter(
        spatial_coords[~mask, 0],
        spatial_coords[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.5
    )

    # Plot this cluster in color
    ax.scatter(
        spatial_coords[mask, 0],
        spatial_coords[mask, 1],
        c=cluster_colors[cluster],
        s=spot_size,
        alpha=1.0
    )

    ax.set_title(f'Leiden {cluster}')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')

# Remove empty subplots
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
#plt.savefig('individual_clusters_spatial.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

adata = bgs5
categ = 'adjusted_L3_transferred'

# 1. EXPLICIT cell type order
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
}

# 2. Keep only cell types that exist in the data
# (Works even if obs[categ] is not categorical)
present = set(adata.obs[categ].dropna().unique())
cell_type_order = [ct for ct in cell_type_order if ct in present]
n_clusters = len(cell_type_order)

# Safety check for missing colors
missing = set(cell_type_order) - set(cell_type_colors)
assert not missing, f"Missing colors for: {missing}"

# 3. Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten()

# 4. Spatial coords + GLOBAL limits (prevents zooming)
spatial = adata.obsm['spatial']
x = spatial[:, 0]
y = spatial[:, 1]
xlim = (np.nanmin(x), np.nanmax(x))
ylim = (np.nanmin(y), np.nanmax(y))

grey_color = '#D3D3D3'
spot_size = 0.04

# 5. Plot each cell type
for idx, cell_type in enumerate(cell_type_order):
    ax = axes[idx]

    mask = (adata.obs[categ] == cell_type).to_numpy()

    # Background (all other cells)   FIXED
    ax.scatter(
        spatial[~mask, 0],
        spatial[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.2,
        rasterized=True
    )

    # Foreground (this cell type)
    ax.scatter(
        spatial[mask, 0],
        spatial[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')

    # Apply identical limits to every subplot   FIXED
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.invert_yaxis()
    ax.axis('off')

# 6. Remove empty panels
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.subplots_adjust(
    wspace=-0.2,   #  less horizontal space between columns
    hspace=0.25    #  more vertical space between rows
)

# plt.savefig('individual_cell_types_spatial.pdf', bbox_inches='tight')  # nice with rasterized points
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# Matplotlib settings for Illustrator-friendly output
# - fonts remain editable (TrueType)
# - rasterized scatter keeps SVGs lightweight
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'   # keep text as text (editable) in SVG

adata = bgs5              # filtered + celltyped (foreground)
adata_bg = bgs5_full      # full (background only)
categ = 'adjusted_L3_transferred'

# 1. EXPLICIT cell type order
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# 2. Explicit colors
cell_type_colors = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',  # unused here, fine to keep
}

# 3. Keep only cell types present in the typed object
present = set(adata.obs[categ].dropna().unique())
cell_type_order = [ct for ct in cell_type_order if ct in present]
n_clusters = len(cell_type_order)

missing = set(cell_type_order) - set(cell_type_colors)
assert not missing, f"Missing colors for: {missing}"

# 4. Spatial coords + global limits (from FULL dataset)
spatial_bg = adata_bg.obsm['spatial']
spatial_fg = adata.obsm['spatial']

xlim = (np.nanmin(spatial_bg[:, 0]), np.nanmax(spatial_bg[:, 0]))
ylim = (np.nanmin(spatial_bg[:, 1]), np.nanmax(spatial_bg[:, 1]))

grey_color = '#D3D3D3'
spot_size_bg = 0.0005
spot_size_fg = 0.15

# 5. Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten()

# 6. Plot each cell type panel
for idx, cell_type in enumerate(cell_type_order):
    ax = axes[idx]
    mask = (adata.obs[categ] == cell_type).to_numpy()

    # Background: ALL cells from bgs5_full in grey (includes low-UMI cells)
    ax.scatter(
        spatial_bg[:, 0],
        spatial_bg[:, 1],
        c=grey_color,
        s=spot_size_bg,
        alpha=0.70,
        rasterized=True
    )

    # Foreground: typed cells from filtered object in color
    ax.scatter(
        spatial_fg[mask, 0],
        spatial_fg[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size_fg,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

# 7. Remove empty panels
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

# Spacing (your preferred padding)
plt.subplots_adjust(
    wspace=-0.2,   # less horizontal padding
    hspace=0.25    # more vertical padding
)

# 8. Save outputs as SVG (Illustrator-friendly)
# # One multi-panel SVG:
# fig.savefig("bgs5_celltype_panels.svg", bbox_inches="tight", dpi=600)

# # Also save one SVG per panel:
# for ax, cell_type in zip(axes[:n_clusters], cell_type_order):
#     fig_single, ax_single = plt.subplots(figsize=(6, 6))

#     mask = (adata.obs[categ] == cell_type).to_numpy()

#     ax_single.scatter(
#         spatial_bg[:, 0], spatial_bg[:, 1],
#         c=grey_color, s=spot_size_bg, alpha=0.20, rasterized=True
#     )
#     ax_single.scatter(
#         spatial_fg[mask, 0], spatial_fg[mask, 1],
#         c=cell_type_colors[cell_type], s=spot_size_fg, alpha=1.0, rasterized=True
#     )

#     ax_single.set_title(cell_type, fontsize=14)
#     ax_single.set_aspect('equal')
#     ax_single.set_xlim(xlim)
#     ax_single.set_ylim(ylim)
#     ax_single.invert_yaxis()
#     ax_single.axis('off')

#     fig_single.savefig(f"bgs5_{cell_type}.svg", bbox_inches="tight", dpi=600)
#     plt.close(fig_single)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# Matplotlib settings for Illustrator-friendly PDF
# - fonts remain editable (TrueType)
# - rasterized scatter keeps PDFs lightweight
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',  # harmless to keep
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# Output folder
outdir = Path("./20260115 - pdf figs/bgs5")
outdir.mkdir(parents=True, exist_ok=True)

# Data
adata = bgs5              # filtered + celltyped (foreground)
adata_bg = bgs5_full      # full (background only)
categ = 'adjusted_L3_transferred'

# 1. EXPLICIT cell type order
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# 2. Explicit colors
cell_type_colors = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',  # unused here, fine to keep
}

# 3. Keep only cell types present in the typed object
present = set(adata.obs[categ].dropna().unique())
cell_type_order = [ct for ct in cell_type_order if ct in present]
n_clusters = len(cell_type_order)

missing = set(cell_type_order) - set(cell_type_colors)
assert not missing, f"Missing colors for: {missing}"

# 4. Spatial coords + global limits (from FULL dataset)
spatial_bg = adata_bg.obsm['spatial']
spatial_fg = adata.obsm['spatial']

xlim = (np.nanmin(spatial_bg[:, 0]), np.nanmax(spatial_bg[:, 0]))
ylim = (np.nanmin(spatial_bg[:, 1]), np.nanmax(spatial_bg[:, 1]))

grey_color = '#D3D3D3'
spot_size_bg = 0.0005
spot_size_fg = 0.12

# 5. Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten()

# 6. Plot each cell type panel
for idx, cell_type in enumerate(cell_type_order):
    ax = axes[idx]
    mask = (adata.obs[categ] == cell_type).to_numpy()

    # Background: ALL cells from bgs5_full in grey (includes low-UMI cells)
    ax.scatter(
        spatial_bg[:, 0],
        spatial_bg[:, 1],
        c=grey_color,
        s=spot_size_bg,
        alpha=1,
        rasterized=True
    )

    # Foreground: typed cells from filtered object in color
    ax.scatter(
        spatial_fg[mask, 0],
        spatial_fg[mask, 1],
        c=cell_type_colors[cell_type],
        s=spot_size_fg,
        alpha=1.0,
        rasterized=True
    )

    ax.set_title(cell_type, fontsize=14)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.invert_yaxis()
    ax.axis('off')

# 7. Remove empty panels
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

# Spacing (your preferred padding)
plt.subplots_adjust(
    wspace=-0.2,   # less horizontal padding
    hspace=0.25    # more vertical padding
)

# 8. Save outputs as PDF (Illustrator-friendly)
# One multi-panel PDF:
fig.savefig(
    outdir / "bgs5_celltype_panels.pdf",
    format="pdf",
    bbox_inches="tight",
    dpi=300,          # raster layer resolution; drop to 200 for smaller files
    transparent=True
)

# Also save one PDF per panel:
for cell_type in cell_type_order:
    fig_single, ax_single = plt.subplots(figsize=(6, 6))
    mask = (adata.obs[categ] == cell_type).to_numpy()

    ax_single.scatter(
        spatial_bg[:, 0], spatial_bg[:, 1],
        c=grey_color, s=spot_size_bg, alpha=1, rasterized=True
    )
    ax_single.scatter(
        spatial_fg[mask, 0], spatial_fg[mask, 1],
        c=cell_type_colors[cell_type], s=spot_size_fg, alpha=1.0, rasterized=True
    )

    ax_single.set_title(cell_type, fontsize=14)
    ax_single.set_aspect('equal')
    ax_single.set_xlim(xlim)
    ax_single.set_ylim(ylim)
    ax_single.invert_yaxis()
    ax_single.axis('off')

    fig_single.savefig(
        outdir / f"bgs5_{cell_type}.pdf",
        format="pdf",
        bbox_inches="tight",
        dpi=300,
        transparent=True
    )
    plt.close(fig_single)

plt.show()
print(f"Saved multi-panel + {len(cell_type_order)} single-panel PDFs to: {outdir}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# Matplotlib settings for Illustrator-friendly PDF
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# Output folder
outdir = Path("./20260115 - pdf figs/bgs5")
outdir.mkdir(parents=True, exist_ok=True)

# Data
adata = bgs5
adata_bg = bgs5_full
categ = 'adjusted_L3_transferred'

# Target cell type (ONLY)
cell_type = 'DRD2-BACH2'

cell_type_colors = {
    'DRD2-BACH2': '#8c564b',
}

# Spatial coordinates
spatial_bg = adata_bg.obsm['spatial']
spatial_fg = adata.obsm['spatial']

xlim = (np.nanmin(spatial_bg[:, 0]), np.nanmax(spatial_bg[:, 0]))
ylim = (np.nanmin(spatial_bg[:, 1]), np.nanmax(spatial_bg[:, 1]))

# Marker sizes (ONLY foreground changed)
grey_color = '#D3D3D3'
spot_size_bg = 0.0005        # unchanged
spot_size_fg = 0.35          # BIGGER highlight dots

# Mask
mask = (adata.obs[categ] == cell_type).to_numpy()

# Plot
fig, ax = plt.subplots(figsize=(6, 6))

# Background (unchanged)
ax.scatter(
    spatial_bg[:, 0],
    spatial_bg[:, 1],
    c=grey_color,
    s=spot_size_bg,
    alpha=1,
    rasterized=True
)

# Foreground (bigger dots)
ax.scatter(
    spatial_fg[mask, 0],
    spatial_fg[mask, 1],
    c=cell_type_colors[cell_type],
    s=spot_size_fg,
    alpha=1.0,
    rasterized=True
)

ax.set_title(cell_type, fontsize=14)
ax.set_aspect('equal')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.invert_yaxis()
ax.axis('off')

# Save
fig.savefig(
    outdir / f"bgs5_{cell_type}.pdf",
    format="pdf",
    bbox_inches="tight",
    dpi=300,
    transparent=True
)

plt.close(fig)

print(f"Saved single-panel PDF for {cell_type} to: {outdir}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Spatial gene-expression panels (CosMx) - CLEAN + PRACTICAL
# Using adata.X directly
# Updates you requested:
#  vmin hard floor = 1  (kills low-end purple haze)
#  smaller dots
#  only 4 plots per row

# Inputs
adata = bgs4_lge_mge

genes = [
    'DRD1', 'DRD2',
    'BACH2', 'EPHA4', 'PENK', 'GAD1', 'NKX2-1', 'EGFR', 'PAX6',
    'OLIG1', 'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4',
    'ESAM', 'FN1', 'DCN',
    'P2RY12', 'CX3CR1'
]

# Plot/layout controls
N_COLS = 4                 # <-- 4 plots per row
FIGSIZE_PER_PANEL = 3.8    # slightly bigger since fewer columns
DOT_SIZE = 0.01            # <-- smaller dots
WSPACE = 0.01
HSPACE = 0.10
TOP = 0.90
CMAP = "viridis"

# Expression scaling controls
NORMAL_FLOOR = 1.0         # <-- vmin hard floor = 1
VMax_PCT = 99              # per-gene vmax for clean contrast
ZSCORE_FLOOR = None        # optional: e.g. -0.5

# Helpers
def _get_dense_gene_from_X(adata, gene):
    if gene not in adata.var_names:
        return None
    x = adata[:, gene].X
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).reshape(-1)

def _prep_gene_list(adata, genes):
    genes_present = [g for g in genes if g in adata.var_names]
    genes_missing = [g for g in genes if g not in adata.var_names]
    if genes_missing:
        print(f"Skipping missing genes ({len(genes_missing)}): {genes_missing}")
    if len(genes_present) == 0:
        raise ValueError("None of the requested genes are in adata.var_names")
    return genes_present

def plot_spatial_gene_grid_X(
    adata,
    genes,
    mode="normal",                # "normal" or "zscore"
    ncols=4,
    figsize_per_panel=3.8,
    dot_size=0.01,
    cmap="viridis",
    normal_floor=1.0,
    vmax_pct=99,
    zscore_floor=None,
    wspace=0.01,
    hspace=0.10,
    top=0.90,
):
    genes_present = _prep_gene_list(adata, genes)

    spatial = adata.obsm["spatial"]
    xlim = (np.nanmin(spatial[:, 0]), np.nanmax(spatial[:, 0]))
    ylim = (np.nanmin(spatial[:, 1]), np.nanmax(spatial[:, 1]))

    base = {g: _get_dense_gene_from_X(adata, g).astype(float) for g in genes_present}

    n = len(genes_present)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_panel * ncols, figsize_per_panel * nrows),
        squeeze=False
    )
    axes = axes.flatten()

    last_sc = None

    for i, g in enumerate(genes_present):
        ax = axes[i]
        v = base[g]

        if mode == "normal":
            v_plot = np.clip(v, normal_floor, None)  # <-- vmin floor = 1
            vmax = np.nanpercentile(v, vmax_pct)
            vmin = normal_floor

        elif mode == "zscore":
            mu = np.nanmean(v)
            sd = np.nanstd(v) + 1e-8
            z = (v - mu) / sd

            if zscore_floor is not None:
                z_plot = np.clip(z, zscore_floor, None)
                vmin = zscore_floor
            else:
                z_plot = z
                vmin = np.nanpercentile(z, 1)

            vmax = np.nanpercentile(z, vmax_pct)
            v_plot = z_plot

        else:
            raise ValueError("mode must be 'normal' or 'zscore'")

        last_sc = ax.scatter(
            spatial[:, 0],
            spatial[:, 1],
            c=v_plot,
            s=dot_size,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            alpha=1.0,
            rasterized=True
        )

        ax.set_title(g, fontsize=13, pad=2)
        ax.set_aspect("equal")
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.invert_yaxis()
        ax.axis("off")

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    cbar = fig.colorbar(last_sc, ax=axes[:n], fraction=0.02, pad=0.005)

    if mode == "normal":
        cbar.set_label(f"adata.X (vmin={normal_floor}, vmax=p{vmax_pct} per gene)", fontsize=12)
        fig.suptitle(
            "Spatial gene expression (CLEAN): adata.X\nHard floor vmin=1 + per-gene 99th percentile vmax",
            fontsize=16,
            y=0.99
        )
    else:
        lab = f"z-score(adata.X) (vmax=p{vmax_pct} per gene)"
        if zscore_floor is not None:
            lab += f", floor={zscore_floor}"
        cbar.set_label(lab, fontsize=12)
        fig.suptitle(
            "Spatial gene expression (CLEAN z-score): z(adata.X)\nPer-gene scaling; optional floor",
            fontsize=16,
            y=0.99
        )

    plt.subplots_adjust(wspace=wspace, hspace=hspace, top=top)
    plt.show()

# VERSION 1: Normal clean view (adata.X) with vmin=1
plot_spatial_gene_grid_X(
    adata=adata,
    genes=genes,
    mode="normal",
    ncols=N_COLS,
    figsize_per_panel=FIGSIZE_PER_PANEL,
    dot_size=DOT_SIZE,
    cmap=CMAP,
    normal_floor=NORMAL_FLOOR,
    vmax_pct=VMax_PCT,
    wspace=WSPACE,
    hspace=HSPACE,
    top=TOP,
)

# VERSION 2: Z-scored clean view (optional)
plot_spatial_gene_grid_X(
    adata=adata,
    genes=genes,
    mode="zscore",
    ncols=N_COLS,
    figsize_per_panel=FIGSIZE_PER_PANEL,
    dot_size=DOT_SIZE,
    cmap=CMAP,
    normal_floor=NORMAL_FLOOR,   # unused in zscore mode
    vmax_pct=VMax_PCT,
    zscore_floor=ZSCORE_FLOOR,
    wspace=WSPACE,
    hspace=HSPACE,
    top=TOP,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Spatial gene-expression panels (CosMx) - adata.X
# FIX for "too much purple / barely any yellow":
#  Use a MUCH more aggressive lower cutoff:
#    - vmin = max( user_floor , per-gene percentile floor )
#    - default: user_floor=1 AND percentile_floor=80th percentile
#      (so only the top ~20% of cells get meaningful color range)
# Also keeps:
#  smaller dots
#  4 plots per row
#  tight spacing
# Notes:
# - This is *visualization-driven* (high contrast). It will make expression look
#   more binary / highlight hotspots, which is what you want if everything looks purple.

# Inputs
adata = bgs4_lge_mge

genes = [
    'DRD1', 'DRD2',
    'BACH2', 'EPHA4', 'PENK', 'GAD1', 'NKX2-1', 'EGFR', 'PAX6',
    'OLIG1', 'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4',
    'ESAM', 'FN1', 'DCN',
    'P2RY12', 'CX3CR1'
]

# Plot/layout controls
N_COLS = 4
FIGSIZE_PER_PANEL = 3.8
DOT_SIZE = 0.008          # even smaller
WSPACE = 0.01
HSPACE = 0.10
TOP = 0.90
CMAP = "viridis"

# Contrast controls (main knobs)
# Absolute floor (still enforced)
ABS_VMIN_FLOOR = 1.0

# Percentile floor for vmin (AGGRESSIVE to kill purple):
# 70 = top 30% lights up, 80 = top 20%, 90 = top 10% (very punchy)
VMIN_PERCENTILE = 80

# vmax percentile (keep 99 to ignore outliers; reduce to 95 if still too dark)
VMAX_PERCENTILE = 99

# Optional: if you want even more yellow, reduce vmax to compress the top end
# VMAX_PERCENTILE = 95

# Helpers
def _get_dense_gene_from_X(adata, gene):
    if gene not in adata.var_names:
        return None
    x = adata[:, gene].X
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).reshape(-1)

def _prep_gene_list(adata, genes):
    genes_present = [g for g in genes if g in adata.var_names]
    genes_missing = [g for g in genes if g not in adata.var_names]
    if genes_missing:
        print(f"Skipping missing genes ({len(genes_missing)}): {genes_missing}")
    if len(genes_present) == 0:
        raise ValueError("None of the requested genes are in adata.var_names")
    return genes_present

def plot_spatial_gene_grid_X_highcontrast(
    adata,
    genes,
    ncols=4,
    figsize_per_panel=3.8,
    dot_size=0.008,
    cmap="viridis",
    abs_vmin_floor=1.0,
    vmin_percentile=80,
    vmax_percentile=99,
    wspace=0.01,
    hspace=0.10,
    top=0.90,
):
    genes_present = _prep_gene_list(adata, genes)

    spatial = adata.obsm["spatial"]
    xlim = (np.nanmin(spatial[:, 0]), np.nanmax(spatial[:, 0]))
    ylim = (np.nanmin(spatial[:, 1]), np.nanmax(spatial[:, 1]))

    n = len(genes_present)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_panel * ncols, figsize_per_panel * nrows),
        squeeze=False
    )
    axes = axes.flatten()

    last_sc = None

    for i, g in enumerate(genes_present):
        ax = axes[i]
        v = _get_dense_gene_from_X(adata, g).astype(float)

        # per-gene aggressive vmin: max(absolute floor, percentile floor)
        vmin_pct = np.nanpercentile(v, vmin_percentile)
        vmin = max(abs_vmin_floor, vmin_pct)

        # per-gene vmax
        vmax = np.nanpercentile(v, vmax_percentile)

        # if gene is super-sparse, vmin can exceed vmax; handle gracefully
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            # fallback: show binary-ish (expressed vs not) by using vmin=0, vmax=max
            vmin = 0.0
            vmax = np.nanmax(v) if np.isfinite(np.nanmax(v)) else 1.0

        last_sc = ax.scatter(
            spatial[:, 0],
            spatial[:, 1],
            c=v,
            s=dot_size,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            alpha=1.0,
            rasterized=True
        )

        ax.set_title(f"{g}\n(vmin=max({abs_vmin_floor}, p{vmin_percentile}))", fontsize=11, pad=2)
        ax.set_aspect("equal")
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.invert_yaxis()
        ax.axis("off")

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    cbar = fig.colorbar(last_sc, ax=axes[:n], fraction=0.02, pad=0.005)
    cbar.set_label(
        f"adata.X (per-gene vmin=max({abs_vmin_floor}, p{vmin_percentile}), vmax=p{vmax_percentile})",
        fontsize=12
    )

    fig.suptitle(
        "Spatial gene expression (HIGH CONTRAST): adata.X\nAggressive vmin percentile to reduce purple and reveal hotspots",
        fontsize=16,
        y=0.99
    )

    plt.subplots_adjust(wspace=wspace, hspace=hspace, top=top)
    plt.show()

# Run HIGH-CONTRAST version
plot_spatial_gene_grid_X_highcontrast(
    adata=adata,
    genes=genes,
    ncols=N_COLS,
    figsize_per_panel=FIGSIZE_PER_PANEL,
    dot_size=DOT_SIZE,
    cmap=CMAP,
    abs_vmin_floor=ABS_VMIN_FLOOR,
    vmin_percentile=VMIN_PERCENTILE,
    vmax_percentile=VMAX_PERCENTILE,
    wspace=WSPACE,
    hspace=HSPACE,
    top=TOP,
)